# Student Performance: Exploratory Data Analysis

This notebook explores student demographics, preparation, and exam performance. It is designed to be rerun from a fresh kernel and ends with evidence-based findings.

Dataset Source = https://www.kaggle.com/datasets/spscientist/students-performance-in-exams?datasetID=74977

## 1. Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")
%matplotlib inline

In [ ]:
# Works when the notebook is opened from the project root, notebook/, or workspace root.
candidates = [Path("data") / "stud.csv", Path("notebook") / "data" / "stud.csv", Path("Student_dashboard") / "notebook" / "data" / "stud.csv"]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/stud.csv. Open the notebook from the project or notebook directory.")
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Initial inspection

Establish the dataset size, schema, data types, and descriptive statistics before analyzing it.

In [ ]:
overview = pd.DataFrame({
    "rows": [df.shape[0]],
    "columns": [df.shape[1]],
    "duplicate_rows": [df.duplicated().sum()],
    "missing_cells": [df.isna().sum().sum()],
})
display(overview)
display(df.dtypes.rename("dtype").to_frame())
display(df.describe(include="all").T)

## 3. Data quality checks and feature engineering

In [ ]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(),
    "dtype": df.dtypes.astype(str),
})
display(quality)
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
score_cols = ["math_score", "reading_score", "writing_score"]
required_cols = score_cols + ["gender", "race_ethnicity", "parental_level_of_education", "lunch", "test_preparation_course"]
missing_required = sorted(set(required_cols) - set(df.columns))
if missing_required:
    raise ValueError(f"Missing expected columns: {missing_required}")
df[score_cols] = df[score_cols].apply(pd.to_numeric, errors="coerce")
df["total_score"] = df[score_cols].sum(axis=1)
df["average_score"] = df[score_cols].mean(axis=1)
df["passed"] = (df["average_score"] >= 40).astype(int)
invalid_scores = ((df[score_cols] < 0) | (df[score_cols] > 100)).sum().sum()
print(f"Invalid score values outside 0–100: {invalid_scores}")
display(df.head())

## 4. Univariate analysis

### Numeric distributions

In [ ]:
numeric_cols = score_cols + ["total_score", "average_score"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(df[col], kde=True, bins=15, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {col.replace('_', ' ').title()}")
    ax.set_xlabel("Score")
    ax.set_ylabel("Students")
axes.flat[-1].axis("off")
fig.suptitle("Score distributions", fontsize=16, y=1.02)
plt.tight_layout()

In [ ]:
display(df[numeric_cols].describe().T.round(2))
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df[numeric_cols], orient="h", ax=ax)
ax.set_title("Score ranges and potential outliers")
ax.set_xlabel("Score")
plt.tight_layout()

### Categorical distributions

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flat, categorical_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, color="cornflowerblue")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Number of students")
    ax.set_ylabel("")
for ax in axes.flat[len(categorical_cols):]:
    ax.axis("off")
fig.suptitle("Student demographic and support categories", fontsize=16, y=1.02)
plt.tight_layout()

## 5. Bivariate analysis

Compare average performance across demographic and support variables. Bars show means with seaborn confidence intervals.

In [ ]:
group_cols = ["gender", "race_ethnicity", "lunch", "test_preparation_course", "parental_level_of_education"]
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, col in zip(axes.flat, group_cols):
    order = df.groupby(col)["average_score"].mean().sort_values(ascending=False).index
    sns.barplot(data=df, x="average_score", y=col, order=order, errorbar="ci", ax=ax, color="teal")
    ax.set_title(f"Average score by {col.replace('_', ' ')}")
    ax.set_xlabel("Average score")
    ax.set_ylabel("")
for ax in axes.flat[len(group_cols):]:
    ax.axis("off")
plt.tight_layout()

In [ ]:
# Group size, mean, median, and pass rate.
for col in group_cols:
    summary = (df.groupby(col, observed=True)
                .agg(students=("average_score", "size"), mean_average=("average_score", "mean"), median_average=("average_score", "median"), pass_rate=("passed", "mean"))
                .sort_values("mean_average", ascending=False))
    summary["pass_rate"] = (summary["pass_rate"] * 100).round(1)
    print(f"\n{col.replace('_', ' ').title()}")
    display(summary.round(2))

## 6. Relationships between subjects

In [ ]:
corr = df[score_cols + ["total_score", "average_score"]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, ax=ax)
ax.set_title("Correlation between scores")
plt.tight_layout()

In [ ]:
pair_cols = score_cols + ["average_score"]
sns.pairplot(df[pair_cols], corner=True, diag_kind="kde")
plt.suptitle("Pairwise relationships among scores", y=1.02)
plt.show()

## 7. Outlier review and findings

The IQR rule identifies unusual observations for review; it does not automatically mean that a record is erroneous.

In [ ]:
outlier_rows = []
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
    outlier_rows.append({"feature": col, "outlier_count": int(mask.sum()), "outlier_percent": round(mask.mean() * 100, 2)})
display(pd.DataFrame(outlier_rows))
gender_means = df.groupby("gender", observed=True)["average_score"].mean().sort_values(ascending=False)
prep_means = df.groupby("test_preparation_course", observed=True)["average_score"].mean().sort_values(ascending=False)
print(f"Highest average score by gender: {gender_means.index[0]} ({gender_means.iloc[0]:.2f})")
print(f"Highest average score by preparation status: {prep_means.index[0]} ({prep_means.iloc[0]:.2f})")
print(f"Overall average score: {df['average_score'].mean():.2f}")
print(f"Overall pass rate (average score >= 40): {df['passed'].mean() * 100:.1f}%")

### Interpretation checklist

- Use group summaries to identify differences, but do not interpret them as causal effects.
- Check group sizes before comparing means; small groups can be unstable.
- Treat `passed` as a simple exploratory threshold, not an official grading policy.
- For decisions, follow up with statistical tests and a clearly defined sampling strategy.